### 1. Setup Environment
Clone the repository and install requirements.

In [ ]:
## --- RUN THIS TO RESET EVERYTHING --- ##

# 1. Clean and Clone
%cd /content
!rm -rf robotic_affordance_project
!git clone https://github.com/francesco-dorati/robotic_affordance_project.git
%cd robotic_affordance_project

# 2. Setup Python Path
import sys
import os
sys.path.append(os.getcwd()) 
!pip install -r requirements.txt

# 3. Mount and Prep Folders
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p data/raw
!mkdir -p data/processed/normals

# 4. Copy and Extract
!echo "Copying & Extracting..."
# Note: Extracting directly from /drive/ is faster than copying then extracting
!tar -xf /content/drive/MyDrive/UMD_Dataset/Affordance\ Dataset\ Tools.tar.gz -C data/raw/ --checkpoint=50000
!tar -xf /content/drive/MyDrive/UMD_Dataset/Affordance\ Dataset\ Clutter.tar.gz -C data/raw/ --checkpoint=50000
# !tar -xf /content/drive/MyDrive/UMD_Dataset/precomputed_normals.tar.gz -C data/processed/ --checkpoint=50000

!echo "Verifying..."
![ "$(ls -A data/raw/part-affordance-dataset/tools/knife_01)" ] && echo "Done!" || echo "Check paths!"

In [ ]:
## --- RUN THIS TO ONLY UPDATE YOUR CODE --- ##

# 1. Move to the repo directory
%cd /content/robotic_affordance_project

# 2. Pull latest changes from GitHub
!git pull origin main

# 3. Force Python to reload your custom modules (Config, Dataset, etc.)
# This ensures that if you changed a file, the notebook actually sees the change.
import importlib
import config
import utils.dataset
import utils.geometry
import models.backbone

importlib.reload(config)
importlib.reload(utils.dataset)
importlib.reload(utils.geometry)
importlib.reload(models.backbone)

print("Code updated and modules reloaded successfully.")

### 2. Visualization

In [ ]:
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from utils.dataset import UMDAffordanceDataset
import config

# 1. Initialize Dataset with dual paths
dataset = UMDAffordanceDataset(
    raw_dir=str(config.RAW_TOOLS),
    crop_size=448
)

# 2. Grab a sample
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)
batch = next(iter(dataloader))

# 3. Plotting
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(batch['rgb'][0].permute(1, 2, 0))
ax[0].set_title("RGB (448x448 Crop)")
ax[1].imshow(batch['mask'][0, 0], cmap='gray')
ax[1].set_title("Affordance Mask")
# Convert normal tensor [-1, 1] to [0, 1] for visualization
norm_vis = (batch['normals'][0].permute(1, 2, 0) + 1.0) / 2.0
ax[2].imshow(norm_vis)
ax[2].set_title("Precomputed Normals")
plt.show()

### 4. Start the Training Loop
Run the training script using the cloud GPU.

In [ ]:
!python scripts/03_train.py